# Amazon Sales Data - Exploratory Data Analysis (EDA)
Bu notebook, ML ile Dinamik Risk ajanımızın temelini olusturacak veriyi anlamlandirmak icin dizayn edilmistir. Orijinal veriye MÜDAHALE EDILMEDEN (read_csv) analizler kurgulanir.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# Veri okunuyor (Sadece Read)
df = pd.read_csv('../database/Amazon Sale Report.csv', low_memory=False)
df.head()

## 1. Veri Kalitesi ve Eksiklik Analizi
Hangi sütunların modeli zehirleyebilecegine ve NaN oranlarina bakalim.

In [ ]:
df.info()
print('\n--- Eksik Veri Oranlari (%) ---')
missing_percent = (df.isnull().sum() / len(df)) * 100
print(missing_percent[missing_percent > 0].sort_values(ascending=False))

## 2. Kategori Çözümlemesi (Status)
Satış başarısı ve Amazon daki genel Status dagilimini gormek (Ornegin Cancelled satislari Demand oranina eklemeli miyiz?)

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, y='Status', order=df['Status'].value_counts().index, palette='viridis')
plt.title('Sipariş Durumlarının Gözlem Sayısı')
plt.tight_layout()
plt.show()

## 3. Zaman Serisi Dağılımı (Trend)
Verinin LSTM, XGBoost veya Prophet ile Demand Forecasting e uygunlugu icin Tarihleri kontrol edelim.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
daily_demand = df.groupby('Date')['Qty'].sum().reset_index()

plt.figure(figsize=(14,4))
sns.lineplot(data=daily_demand, x='Date', y='Qty', color='#c0392b')
plt.title('Zaman Serisi: Günlük Genel Sipariş Adetleri (Trend)')
plt.ylabel('Satış (Qty)')
plt.xlabel('Tarih')
plt.show()

**Makine Öğrenmesi Yorumu:**
Zaman serisinde ciddi sivrilmeler (Spike) varsa model olarak Linear algoritmalar çöker. XGBoost veya Robust algoritmalar şarttır.

In [ ]:
# Amount ve Qty Korelasyonuna Isı Haritasi (Heatmap) Yaklasimi
numeric_cols = ['Qty', 'Amount']
fig, ax = plt.subplots(figsize=(6,4))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=ax)
plt.title('Miktar & Gelir Korelasyonu')
plt.show()